# Chapter 6: planned against actual; the missing movements

Section 6.5 lists planned against actual per brew and flags the movements that are missing. The plan is the
ERP order. The actual is the brew sheet. Malt costs USD 0.90 a kg (synthetic).

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import kestrel as k

MALT_USD_PER_KG = 0.90
order = k.load("bronze_erp_order")
sheet = k.load("bronze_brewsheet_brew")
both = order.merge(sheet, on="brew_id")
both["actual_grist_kg"] = both.actual_grist_t * 1000
both["variance_kg"] = both.actual_grist_kg - both.planned_grist_kg
both["variance_pct"] = (100 * both.variance_kg / both.planned_grist_kg).round(2)
both["variance_usd"] = both.variance_kg * MALT_USD_PER_KG
print(both[["brew_id", "planned_grist_kg", "actual_grist_kg", "variance_kg", "variance_pct", "variance_usd",
            "planned_wort_hl_x", "actual_wort_hl"]].to_string(index=False))

What the ERP posted for the same malt. The ERP backflushes the plan, so a brew that took more than planned
still shows the planned weight. Keep both columns; neither overwrites the other.

In [ ]:
gm = k.load("bronze_erp_goods_movement")
factor = k.load("unit_factor").set_index("from_unit").factor
malt_codes = {"100234", "100240"}
posted = gm[gm.item_code.isin(malt_codes)].copy()
posted["kg"] = posted.quantity * posted.unit.map(factor)
posted = posted.groupby("order_ref").kg.sum().rename("erp_malt_kg").reset_index()
check = both.merge(posted, on="order_ref")[["brew_id", "actual_grist_kg", "erp_malt_kg"]]
check["gap_kg"] = check.actual_grist_kg - check.erp_malt_kg
print(check.to_string(index=False))

The movements each brew and each pack lot should cause, against the movements the model holds.

In [ ]:
con = k.connect()
mv = k.q(con, "SELECT m.*, i.item_type FROM movement m JOIN item i USING (item_key)")
brews = k.q(con, "SELECT brew_key, brew_id FROM brew")
pack = k.q(con, "SELECT lot_key, lot_id FROM lot WHERE lot_type = 'pack'")
missing = []
for _, b in brews.iterrows():
    into = mv[(mv.to_node_type == "brew") & (mv.to_node_key == b.brew_key)]
    out = mv[(mv.from_node_type == "brew") & (mv.from_node_key == b.brew_key)]
    for need, ok in [("malt issue", (into.item_type == "malt").any()), ("hop issue", (into.item_type == "hops").any()),
                     ("wort to fermenter", (out.movement_type == "produce").any())]:
        if not ok:
            missing.append((b.brew_id, need))
for _, p in pack.iterrows():
    into = mv[(mv.to_node_type == "pack_lot") & (mv.to_node_key == p.lot_key)]
    needs = [("beer from bright tank", (into.item_type == "product").any()),
             ("container issue", into.item_type.isin(["can", "bottle"]).any())]
    if "-B" in p.lot_id:
        needs.append(("crown issue", (into.item_type == "crown").any()))
    for need, ok in needs:
        if not ok:
            missing.append((p.lot_id, need))
print(pd.DataFrame(missing, columns=["node", "missing movement"]).to_string(index=False))

Section 6.4's worked set for brew B26-0068 and the can run it fed, read back from the movement table.

In [ ]:
edges = k.q(con, "SELECT * FROM edge")
print(edges[(edges.from_node == "M-26-0087") & (edges.to_node == "B26-0068")][["from_node", "to_node", "quantity", "unit"]])
print(edges[edges.from_node == "B26-0068"][["from_node", "to_node", "quantity", "unit"]])
can_run = edges[edges.to_node == "L260412-C1-1"]
print(can_run[["from_node", "to_node", "quantity", "unit"]])
cans = can_run[can_run.unit == "each"].quantity.iloc[0]
hl = can_run[can_run.unit == "hL"].quantity.iloc[0]
print(f"cans cost {cans * 0.14:,.0f} USD, {cans * 0.14 / hl:.1f} USD a hL")